#### Data ingestion to vector DB pipeline

In [3]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
# from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [5]:
## read all the pdfs inside the directory
from pathlib import Path
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 3 PDF files to process

Processing: Attention.pdf
  ✓ Loaded 1 pages

Processing: Embeddings.pdf
  ✓ Loaded 6 pages

Processing: objectDetection.pdf
  ✓ Loaded 2 pages

Total documents loaded: 9


In [6]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs


In [7]:
chunks=split_documents(all_pdf_documents)
chunks

Split 9 documents into 17 chunks

Example chunk:
Content: Attention
 
in
 
deep
 
learning
 
is
 
a
 
mechanism
 
that
 
allows
 
models
 
to
 
dynamically
 
focus
 
on
 
the
 
most
 
relevant
 
parts
 
of
 
an
 
input
 
sequence
 
by
 
assigning
 
varying
 ...
Metadata: {'producer': 'Skia/PDF m156 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Attention', 'source': '..\\data\\pdf\\Attention.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Attention.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Skia/PDF m156 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Attention', 'source': '..\\data\\pdf\\Attention.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Attention.pdf', 'file_type': 'pdf'}, page_content='Attention\n \nin\n \ndeep\n \nlearning\n \nis\n \na\n \nmechanism\n \nthat\n \nallows\n \nmodels\n \nto\n \ndynamically\n \nfocus\n \non\n \nthe\n \nmost\n \nrelevant\n \nparts\n \nof\n \nan\n \ninput\n \nsequence\n \nby\n \nassigning\n \nvarying\n \nimportance\n \nweights\n \nto\n \ndifferent\n \nelements.\n \nWhy\n \nAttention\n \nWas\n \nCreated\n \nTraditional\n \nmodels\n \nlike\n \nRecurrent\n \nNeural\n \nNetworks\n \n(RNNs)\n \nand\n \nLSTMs\n \nprocess\n \ntext\n \nor\n \nsequential\n \ndata\n \nstep-by-step.\n \nThey\n \ncompress\n \nentire\n \ninputs\n \ninto\n \na\n \nsingle\n \nfixed-size\n \nvector.\n \nThis\n \ncreates\n \nan\n \ninformation\n \nbottleneck,\n \ncausing\n \nthe\n \nmodel\

In [8]:
### Embedding

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
        
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """ Load the SentenceTransformer model"""
        try:
            self.model = SentenceTransformer(self.model_name)
            print("Model loaded successfully, embedding dimentions : ", self.model.get_sentence_embedding_dimension())
        except Exception as e:
            print(f"Failed to load model: {e}")

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a given text using the loaded model.
        
        Args:
            text: The input text for which embeddings are to be generated.
        
        Returns:
            A numpy array containing the generated embeddings.

        """
        if not self.model:
            raise ValueError("Model not loaded")

        embeddings = self.model.encode(texts, show_progress_bar = True)
        print(f"Generated embeddings with shape : {embeddings.shape}")

        return embeddings

    def get_embedding_dimention(self) -> int:
        """Get the embedding dimention of the model"""
        if not self.model:
            raise ValueError("Model not loaded")

        return self.model.get_embedding_dimension()
